In [5]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
import random
import matplotlib.pyplot as plt

# Set global seed for reproducibility
np.random.seed(0)
random.seed(0)

# Load and preprocess data
def load_CSN_data():
    csv_path = '../CSN_Viability_Database_2025.csv'
    return pd.read_csv(csv_path)

CSN = load_CSN_data()

# Drop unused columns
CSN = CSN.drop(['Example ID', 'Source', 'Figure ID', 'Data Provider', 'PI',
                'Date Received', 'Data Measurment Published', 'Prior Exposure', 'Comments', 'Error'], axis=1)

# One-hot encode categorical variables
CSN_prepared = pd.get_dummies(CSN, dtype=int)

# Add engineered features
CSN_prepared['Surface Area per Liter'] = CSN_prepared['Surface Area (NMC) (m2/g)'] * CSN_prepared['Concentration (mg/L)']
CSN_prepared = CSN_prepared.drop(['Surface Area (NMC) (m2/g)'], axis=1)
CSN_prepared['log Concentration'] = np.log10(CSN_prepared['Concentration (mg/L)'] + 1e-9)
CSN_prepared = CSN_prepared.drop(['Concentration (mg/L)'], axis=1)

# Split data
CSN_new = CSN_prepared[-38:]
CSN_prepared = CSN_prepared.drop(CSN_prepared.index[-38:])
X_train = CSN_prepared.drop(['Viability_Fraction'], axis=1)
Y_train = CSN_prepared['Viability_Fraction']
X_test = CSN_new.drop(['Viability_Fraction'], axis=1)
Y_test = CSN_new['Viability_Fraction']

# ----------------------------------------
# Fixed Random Forest parameters
# ----------------------------------------
rf_params = {
    'n_estimators': 100,
    'max_depth': 7,
    'min_samples_split': 3,
    'min_samples_leaf': 1,
}

# ----------------------------------------
# Run 100 times with different random seeds
# ----------------------------------------
mae_list = []
random_seeds = random.sample(range(0, 100000), 100)

for seed in random_seeds:
    model = RandomForestRegressor(random_state=seed, **rf_params)
    model.fit(X_train, Y_train)
    y_pred = model.predict(X_test)
    test_mae = mean_absolute_error(Y_test, y_pred)
    mae_list.append(test_mae)
    print(f"Seed {seed}: MAE = {test_mae:.4f}")

# Summary statistics
mae_array = np.array(mae_list)
print("\nSummary of Random Forest predictions over 100 random seeds:")
print("Mean MAE: {:.4f} ± {:.4f}".format(mae_array.mean(), mae_array.std()))
print("Min MAE: {:.4f}".format(mae_array.min()))
print("Max MAE: {:.4f}".format(mae_array.max()))


Seed 50494: MAE = 0.2706
Seed 99346: MAE = 0.2961
Seed 55125: MAE = 0.3069
Seed 5306: MAE = 0.2966
Seed 33936: MAE = 0.2839
Seed 67013: MAE = 0.2764
Seed 63691: MAE = 0.2982
Seed 53075: MAE = 0.3030
Seed 39755: MAE = 0.3077
Seed 62468: MAE = 0.3056
Seed 46930: MAE = 0.2864
Seed 76465: MAE = 0.2924
Seed 28631: MAE = 0.2802
Seed 66150: MAE = 0.2811
Seed 18254: MAE = 0.2897
Seed 36941: MAE = 0.2919
Seed 18316: MAE = 0.2875
Seed 99064: MAE = 0.2971
Seed 12429: MAE = 0.2921
Seed 81050: MAE = 0.2954
Seed 32834: MAE = 0.2965
Seed 69804: MAE = 0.2876
Seed 92428: MAE = 0.3035
Seed 78892: MAE = 0.2836
Seed 19262: MAE = 0.2895
Seed 40651: MAE = 0.2912
Seed 12945: MAE = 0.2947
Seed 95660: MAE = 0.2906
Seed 9665: MAE = 0.2859
Seed 89651: MAE = 0.3011
Seed 43279: MAE = 0.2818
Seed 61884: MAE = 0.2968
Seed 73375: MAE = 0.2787
Seed 13199: MAE = 0.2785
Seed 46372: MAE = 0.2846
Seed 56907: MAE = 0.2918
Seed 41444: MAE = 0.2900
Seed 80070: MAE = 0.2953
Seed 83941: MAE = 0.3074
Seed 26801: MAE = 0.2871
Se